# 바닐라(vanilla) Baseline — before→after 스토리용

**목적**: 아무 도메인 처리 없이 **기본 ML + 전체 피처**로 성능을 뽑아 "before"를 고정한다.
lean-85(after)와 **동일한 train/valid/test 방식·동일 RMSE**로 비교 가능하게 맞춘다. · 제작 2026-07-19

> **미러 미검증본**(v14 전달 방식) — 코드는 정적 검토(구문·컬럼 처리)만 거쳤고 **성능 수치는 로컬 실행으로 확정**한다.
> 첫 실행 오류 시 메시지 주면 즉시 수정.

## 무엇이 '바닐라'인가 (의도적으로 순진하게)
- **전처리 없음**: row→웨이퍼(C64)는 수치=평균, 범주=첫값. 그 외 도메인 가공(slope·PM·레짐·상호작용) 일절 없음.
- **전체 피처**: 명세상 제외/집계 규칙 무시하고 남는 컬럼 전부 사용. 범주형은 label 인코딩(무관측=NaN).
- **기본 ML**: `LinearRegression()` · `LGBMRegressor(random_state=42)` — **튜닝 없이 디폴트**.

## 2×2 매트릭스 (핵심 대비축)
| 피처 범위 | 의미 |
|---|---|
| **ID포함** | Lot/중복WF ID(C20·C21·C22·C34·C35·C38)까지 그대로 투입 — valid Lot의 99.9%가 train에 존재 → **누수로 valid/test가 가짜로 좋아 보임** |
| **ID제외** | 식별자만 뺀 정직 바닐라 — before의 실제 출발선 |

각 범위 × {Linear, LGBM} = **4 구성**. 지표: `train_insample`(과적합 참고) · `cv5_random`(랜덤 KFold, 낙관) · `cv5_gkf_c20`(**GroupKFold C20 = lean-85와 동일 축**) · `valid`·`test`(홀드아웃 직접 채점) RMSE.

**세 축 주의**: (1) `valid`/`test` = same-era 홀드아웃 → v8/v10과 직접 비교. (2) `cv5_gkf_c20` = lean-85 lot-CV(66.96)와 동일 축. (3) `cv5_random` = lot-mate 누수로 낙관 → GKF와 비교 금지(순진한 CV 시연).

**출력**: `baseline_vanilla_results.csv` (노트북 폴더). 실행 순서: 셀1 설정 → 셀2 로드/정리 → 셀3 웨이퍼 집계 → 셀4 매트릭스 빌더 → 셀5 학습·평가(2 CV축) → 셀6 결과표.

In [1]:
# [셀1] import + 설정
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

def find_root():
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (base / "문제1(하)").exists():
            return base
    raise FileNotFoundError("'문제1(하)' 폴더를 못 찾음 — 프로젝트 루트나 그 하위에서 실행하세요")

ROOT = find_root(); DATA = ROOT / "문제1(하)"; ANS = ROOT / "문제1_하_answer"
WF, TARGET, TIME = "C64", "C65", "C40"
ID_COLS = ["C20", "C21", "C22", "C34", "C35", "C38"]   # Lot ID + 중복 WF ID (누수 후보)
SEED = 42

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

print("ROOT:", ROOT, "| lightgbm", lgb.__version__)

ROOT: C:\Users\Dell3571\Documents\defect_test_prediction_MK | lightgbm 4.6.0


In [2]:
# [셀2] 데이터 로드 + test_X 정리(중복 C64·빈 C65 제거) + 정답
train = pd.read_csv(DATA / "train_data.csv", low_memory=False)
valid = pd.read_csv(DATA / "valid_X.csv", low_memory=False)
test  = pd.read_csv(DATA / "test_X.csv",  low_memory=False)

# ⚠️ test_X 는 C64 가 중복(C64.1)이고 빈 C65 열이 붙어 있음 → 제거
test = test.drop(columns=[c for c in ["C64.1", TARGET] if c in test.columns])

vY = pd.read_csv(ANS / "valid_Y_answer.csv")   # [C64, C65]
tY = pd.read_csv(ANS / "test_Y_answer.csv")
train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")   # 혼합형 방어

print(f"train {train.shape} | valid {valid.shape} | test {test.shape}")
print(f"wafers  train {train[WF].nunique()} | valid {valid[WF].nunique()} | test {test[WF].nunique()}")
print(f"정답 커버  valid {vY[WF].nunique()}/{valid[WF].nunique()} | test {tY[WF].nunique()}/{test[WF].nunique()}")

train (123614, 65) | valid (20577, 64) | test (20510, 64)
wafers  train 11939 | valid 1990 | test 1990
정답 커버  valid 1990/1990 | test 1990/1990


In [3]:
# [셀3] row → 웨이퍼(C64) 집계 : 수치=평균, 범주=첫값 (무처리 바닐라)
def to_wafer(df):
    num = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
    obj = [c for c in df.select_dtypes(include=["object"]).columns if c != WF]
    g = df.groupby(WF, sort=False)
    parts = [g[num].mean()]
    if obj:
        parts.append(g[obj].first())
    out = pd.concat(parts, axis=1)
    if TARGET in df.columns:
        out[TARGET] = g[TARGET].first()
    return out.reset_index()

trW = to_wafer(train); vaW = to_wafer(valid); teW = to_wafer(test)
trW = trW[trW[TARGET].notna()].reset_index(drop=True)         # 타깃 결측 웨이퍼 제외
print("집계 후(웨이퍼 단위):", trW.shape, vaW.shape, teW.shape)

집계 후(웨이퍼 단위): (11939, 65) (1990, 64) (1990, 64)


In [4]:
# [셀4] 피처행렬 빌더 — 범위(ID포함/제외) 선택 + label 인코딩(train 기준)
# ⚠️ pandas 버전마다 범주형 dtype이 object/category/string 로 갈려 'dtype==object' 판정은 불안정.
#    → is_numeric_dtype 로 '수치가 아니면 전부' 인코딩하고, 끝에 수치 강제(안전망)해 잔여 문자열 0 보장.
def build_matrix(use_ids):
    drop = [WF, TARGET] + ([] if use_ids else ID_COLS)
    feats = [c for c in trW.columns if c not in drop]
    feats = [c for c in feats if c in vaW.columns and c in teW.columns]   # 3분할 공통 컬럼만
    Xtr, Xva, Xte = trW[feats].copy(), vaW[feats].copy(), teW[feats].copy()
    for c in feats:                                    # 비수치(범주/문자) → train 기준 정수코드(무관측=NaN)
        if not pd.api.types.is_numeric_dtype(Xtr[c]):
            cats = {v: i for i, v in enumerate(pd.unique(Xtr[c].dropna()))}
            Xtr[c] = Xtr[c].map(cats)
            Xva[c] = Xva[c].map(cats)
            Xte[c] = Xte[c].map(cats)
    # 안전망: 남은 비수치까지 수치 강제(코드화 안 된 잔여만 NaN) → LinearRegression 문자열 오류 원천 차단
    Xtr = Xtr.apply(pd.to_numeric, errors="coerce").astype("float64")
    Xva = Xva.apply(pd.to_numeric, errors="coerce").astype("float64")
    Xte = Xte.apply(pd.to_numeric, errors="coerce").astype("float64")
    assert all(pd.api.types.is_numeric_dtype(Xtr[c]) for c in feats), "비수치 피처 잔존"
    return feats, Xtr, Xva, Xte

# 정답을 웨이퍼 순서에 정렬
yva = vaW[WF].map(vY.set_index(WF)[TARGET]).to_numpy()
yte = teW[WF].map(tY.set_index(WF)[TARGET]).to_numpy()
ytr = trW[TARGET].reset_index(drop=True)
assert not np.isnan(yva).any() and not np.isnan(yte).any(), "정답 매칭 누락"
print("정답 정렬 완료 | valid", len(yva), "test", len(yte))

정답 정렬 완료 | valid 1990 test 1990


In [5]:
# [셀5] 학습·평가 — 2(범위) × 2(모델), 튜닝 없는 디폴트
# 두 CV 축을 나란히: cv5_random(랜덤 KFold, 낙관) vs cv5_gkf_c20(GroupKFold C20, 정직=lean-85와 동일 축)
from sklearn.model_selection import GroupKFold

def make_model(kind):
    return LinearRegression() if kind == "linear" else lgb.LGBMRegressor(random_state=SEED, n_jobs=-1)

def _prep(kind, tr, *others):
    # LinearRegression 은 NaN 불가 → train 중앙값 대치(스케일링 안 함 = 바닐라). LGBM 은 NaN 그대로.
    if kind != "linear":
        return (tr, *others)
    med = tr.median(numeric_only=True)
    return tuple(x.fillna(med).fillna(0) for x in (tr, *others))

def cv5_random(kind, X, y):
    kf = KFold(5, shuffle=True, random_state=SEED); sc = []
    for tri, vai in kf.split(X):
        a, b = _prep(kind, X.iloc[tri], X.iloc[vai])
        m = make_model(kind); m.fit(a, y.iloc[tri]); sc.append(rmse(y.iloc[vai], m.predict(b)))
    return float(np.mean(sc))

def cv5_gkf(kind, X, y, groups):
    # GroupKFold(C20): 같은 Lot 웨이퍼가 fold 를 가로지르지 않음 → lot-mate 누수 차단(정직).
    # lean-85 lot-CV(66.96)와 동일 축. 셔플 없어 결정론적(seed 무관).
    oof = np.empty(len(y))
    for tri, vai in GroupKFold(5).split(X, y, groups):
        a, b = _prep(kind, X.iloc[tri], X.iloc[vai])
        m = make_model(kind); m.fit(a, y.iloc[tri]); oof[vai] = m.predict(b)
    return rmse(y, oof)

groups = trW["C20"].reset_index(drop=True)          # Lot ID — GKF 그룹키(피처 아님)
assert len(groups) == len(ytr), "groups-ytr 정렬 불일치"

rows = []
for use_ids in [True, False]:
    feats, Xtr, Xva, Xte = build_matrix(use_ids)
    for kind in ["linear", "lgbm"]:
        a, va, te = _prep(kind, Xtr, Xva, Xte)
        m = make_model(kind); m.fit(a, ytr)
        row = dict(scope="ID포함" if use_ids else "ID제외", model=kind, n_feats=len(feats),
                   train_insample=round(rmse(ytr, m.predict(a)), 3),
                   cv5_random=round(cv5_random(kind, Xtr, ytr), 3),
                   cv5_gkf_c20=round(cv5_gkf(kind, Xtr, ytr, groups), 3),
                   valid=round(rmse(yva, m.predict(va)), 3),
                   test=round(rmse(yte, m.predict(te)), 3))
        rows.append(row); print(row)

{'scope': 'ID포함', 'model': 'linear', 'n_feats': 63, 'train_insample': 81.5, 'cv5_random': 81.689, 'cv5_gkf_c20': 82.296, 'valid': 1545.561, 'test': 1542.193}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7558
[LightGBM] [Info] Number of data points in the train set: 11939, number of used features: 43
[LightGBM] [Info] Start training from score 973.208309
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001622 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7512
[LightGBM] [Info] Number of data points in the train set: 9551, number of used features: 43
[LightGBM] [Info] Start training from score 973.626950
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001550 seconds.
You can set `force_col_wise=true` to remove the ove

In [6]:
# [셀6] 결과표 저장 + 요약 (두 CV 축)
res = pd.DataFrame(rows)[["scope", "model", "n_feats", "train_insample",
                          "cv5_random", "cv5_gkf_c20", "valid", "test"]]
out = Path.cwd() / "baseline_vanilla_results.csv"
res.to_csv(out, index=False)
print("저장:", out)
print("\n[축 주의] cv5_random(랜덤 KFold)은 lot-mate 누수로 낙관 → lean-85 GKF 66.96 과 비교 금지.")
print("          lean-85 와 같은 축 = cv5_gkf_c20 (GroupKFold C20). valid/test 는 홀드아웃 직접 채점(별도 축).")
res

저장: C:\Users\Dell3571\Documents\defect_test_prediction_MK\baseline_vanilla\baseline_vanilla_results.csv

[축 주의] cv5_random(랜덤 KFold)은 lot-mate 누수로 낙관 → lean-85 GKF 66.96 과 비교 금지.
          lean-85 와 같은 축 = cv5_gkf_c20 (GroupKFold C20). valid/test 는 홀드아웃 직접 채점(별도 축).


,scope,model,n_feats,train_insample,cv5_random,cv5_gkf_c20,valid,test
0,ID포함,linear,63,81.500,81.689,82.296,1545.561,1542.193
1,ID포함,lgbm,63,47.786,58.166,73.645,101.044,92.771
2,ID제외,linear,57,86.731,86.870,87.438,434.420,433.041
3,ID제외,lgbm,57,48.968,60.048,69.896,87.318,84.965


## 읽는 법 · before→after 연결

- **before 출발선 = `ID제외` 두 줄** (식별자 없는 정직 바닐라). 이게 lean-85(after)로 얼마나 개선됐는지의 기준점.
- **`ID포함`은 함정 시연**: valid Lot의 99.9%가 train에 존재해, 트리 모델(LGBM)이 Lot 코드로 웨이퍼 레벨을 외워 valid/test가 **가짜로 낮게** 나올 수 있음. 이게 "왜 프로젝트 원칙이 Lot ID 금지인가"의 근거.
- **cv5_random vs cv5_gkf_c20 격차 = 누수 시연**: 랜덤 KFold는 같은 Lot 웨이퍼가 fold 를 가로질러 lot-mate 를 외움 → 낙관. GroupKFold(C20)로 막으면 값이 오름. 이 격차가 프로젝트가 GKF(C20)를 쓴 이유.
- **lean-85와 같은 축 = `cv5_gkf_c20`**. lean-85 lot-CV 66.96 과 이 열을 비교(둘 다 GKF C20 train-CV). `cv5_random`·`valid`/`test`와는 축이 다르니 섞지 말 것.

**주의(정직)**: (1) `train_insample`은 과적합 지표일 뿐 성능 아님. (2) C40(시각)은 범주 코드로 처리돼 valid/test에선 대부분 무관측(NaN)→기여 미미. (3) 이 노트북은 same-era(대회축) 비교용 — lean-85의 현업 수치는 **시간축 재학습 99.84**이며 축이 다름을 명시할 것. (4) 수치는 로컬 실행으로 확정(미러 미검증).